# NanoLM — three trained Transformers, from a fresh Colab session

A curated AI Engineering submission demo. We built a decoder-only Transformer
from explicit Q/K/V projections, scaled causal attention, residual blocks and
next-token cross-entropy. These three models learned Tiny Shakespeare from scratch.
No pretrained model or research training is run here.

**Selected states:** E02 Compact5000, B01 Balanced3000, F01 Quality1750.
This notebook verifies public downloads, actual model shapes, causality, trained
attention and genuine generation. CPU is sufficient. Choose **Runtime → Run all**
in a fresh CPU runtime with internet. No token, Drive mount or developer PC needed.

Small-corpus language models can repeat dialogue, misspell words and emit invalid
UTF-8 bytes. Better validation BPB does not guarantee coherent writing.

## 1. Inspect the native runtime
Record first; do not replace Colab’s native PyTorch or assume the research pins fit it.

In [ ]:
import sys, platform, json, importlib.metadata as metadata, time
from pathlib import Path
STARTED = time.perf_counter()
native = {"python": sys.version, "os": platform.platform()}
for package in ["torch", "numpy", "tokenizers"]:
    try: native[package] = metadata.version(package)
    except metadata.PackageNotFoundError: native[package] = "absent"
print(json.dumps(native, indent=2))

## 2. Clone the exact source
The reviewed scientific base is aa4bb5f. The following pinned submission helper commit adds wrappers/tests only. Its review status is documented in the submission report.

In [ ]:
import subprocess, os
SOURCE = "7ba4d22dcd3818184a3495dc693ff7321ea99076"
REPOSITORY = "https://github.com/rayyan-merchant/Nano-Language-Model.git"
WORK = Path("/content/NanoLM-submission")
ENV = Path("/content/nanolm-submission-env")
PY = ENV / "bin/python"
command_env = dict(os.environ)
command_env.update({"GIT_TERMINAL_PROMPT": "0", "GIT_CONFIG_GLOBAL": os.devnull,
                    "GIT_CONFIG_NOSYSTEM": "1", "HF_HUB_DISABLE_IMPLICIT_TOKEN": "1"})
def run(argv, cwd=None, tail=2500):
    result = subprocess.run([str(x) for x in argv], cwd=cwd, env=command_env,
                            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout[-tail:])
    if result.returncode: raise RuntimeError(f"Command failed (exit {result.returncode}); stop and inspect output")
    return result.stdout
if WORK.exists(): raise FileExistsError("Use a fresh Colab runtime; preserve earlier evidence")
run(["git", "clone", REPOSITORY, WORK])
run(["git", "checkout", SOURCE], cwd=WORK)
print("Exact source:", run(["git", "rev-parse", "HEAD"], cwd=WORK).strip())
assert not run(["git", "status", "--porcelain"], cwd=WORK).strip()

## 3. Provision the compatible CPU interpreter
Isolated Python3.11.14 and torch2.5.1+cpu; NumPy1.26.4 and tokenizers0.21.0. Native Colab remains installed. Installation downloads about a few hundred MB; no GPU is allocated. The package is installed normally, not editable.

In [ ]:
run([sys.executable, "-m", "pip", "install", "uv==0.12.7"])
run([sys.executable, "-m", "uv", "venv", "--python", "3.11.14", "--seed", ENV])
run([PY, "-m", "pip", "install", "torch==2.5.1", "--index-url", "https://download.pytorch.org/whl/cpu"])
run([PY, "-m", "pip", "install", "-r", "requirements.txt", "-r", "requirements-studio.txt"], cwd=WORK)
run([PY, "-m", "pip", "install", ".", "--no-deps"], cwd=WORK)
run([PY, "-m", "pip", "check"], cwd=WORK)
EVIDENCE = WORK / "outputs/submission-colab"
EVIDENCE.mkdir(parents=True)
(EVIDENCE / "native-environment.json").write_text(json.dumps(native, indent=2))

## 4. Import the real implementation
The notebook calls the installed package through its isolated interpreter. It does not copy Transformer code into notebook cells.

In [ ]:
def section(name):
    run([PY, "-m", "tools.submission_demo", "--artifacts", WORK / "model_artifacts",
         "--section", name, "--output", EVIDENCE / name], cwd=WORK, tail=1500)
    return json.loads((EVIDENCE / name / "receipt.json").read_text())
environment = section("environment")
print(json.dumps(environment["environment"], indent=2))

## 5. Numerical, causal and tiny-overfit checks
The existing trusted fixture uses a tiny synthetic sequence and60 optimizer updates. It is a bounded debugging test, not Shakespeare research training. Attention/scaling, split/merge, positions, shifted targets, future-token exclusion and finite gradients are also tested.

In [ ]:
test_log = run([PY, "-m", "pytest", "tests/test_core.py", "tests/test_submission.py", "-v"], cwd=WORK, tail=7000)
(EVIDENCE / "core-tests.txt").write_text(test_log)

## 6. Retrieve all three public packages
Anonymous HTTPS, immutable HF revision, exact file-size/SHA256 and selected-identity checks. This stops on failure; no cached or random weights are substituted. Original research checkpoint hashes remain separate from derived safetensors hashes.

In [ ]:
run([PY, "-m", "tools.submission_models", "--output", WORK / "model_artifacts"], cwd=WORK, tail=5000)
download = json.loads((WORK / "model_artifacts/download_receipt.json").read_text())
print("Verified files:", len(download["files"]), "at", download["revision"])

## 7. Inspect the actually loaded architectures
Parameter counts come from loaded PyTorch objects. Historical BPB is previously measured validation evidence, not likelihood measured in this notebook.

In [ ]:
from IPython.display import display, Markdown, Image
models = section("models")
rows = ["|Model|Selected update|Actual parameters|Blocks|Width|Position|Historical validation BPB|", "|---|---:|---:|---:|---:|---|---:|"]
for m in models["models"]:
    c = m["architecture"]
    rows.append(f"|{m['id'].upper()}|{m['selected_step']}|{m['parameters']:,}|{c['n_layers']}|{c['d_model']}|{c['position']}|{m['historical_validation_bpb']:.8f}|")
display(Markdown("\n".join(rows)))

## 8. Real Byte-BPE tokenization
The canonical train-only1,024-entry tokenizer is hash verified. Generation crops its history to the final128 tokens and resets local positions consistently with training. Byte-level decoding can replace invalid generated UTF-8 with U+FFFD; it does not repair language.

In [ ]:
print(json.dumps(models["tokenizer"], indent=2, ensure_ascii=False))
assert models["tokenizer"]["vocabulary"] == 1024
assert models["tokenizer"]["decoded"] == models["tokenizer"]["prompt"]

## 9. Trained attention and future-token invariance
B01 step3000, layer0/head0. The heatmap uses `return_attention=True` on the actual selected weights. Rows are queries, columns keys. A changed future suffix must not change earlier logits; probabilities above the diagonal must be exactly zero. This is a trained-attention demonstration, not the later complete assignment figure audit.

In [ ]:
audited = section("attention")
a = audited["attention"]
print(json.dumps({k: v for k, v in a.items() if k != "probabilities"}, indent=2))
display(Image(filename=str(EVIDENCE / "attention/trained_attention.png")))

## 10. Genuine generation — all three models
Five fixed prompts × greedy/sampling × three models =30 cases,32 new tokens each, temperature0.7, seed42. Complete raw token IDs and unedited decoded outputs are saved in the receipt. Below are the three required prompts in sampling mode; all30 remain available. Exact cross-platform token equality is measured where prior reference cases exist, not assumed.

In [ ]:
generated = section("generation")
for row in generated["generations"]:
    if row["prompt"] in ["To be, or not to be", "ROMEO:\n", "JULIET:\n"] and not row["greedy"]:
        print("=" * 65)
        print(row["model"].upper(), "step", row["checkpoint_step"], "sampling T=0.7 seed42 new_tokens32", f"{row['seconds']:.3f}s")
        print(row["full_output"])
reference = [r for r in generated["generations"] if r["reference_exact_ids"] is not None]
print("Exact raw-ID reference matches:", sum(r["reference_exact_ids"] for r in reference), "/", len(reference))
print("Full output/token receipt:", EVIDENCE / "generation/receipt.json")

## 11. Compare inference cost and historical validation
These are actual32-token CPU generation measurements in this session, not training runtimes or robust hardware benchmarks. One seed provides no variance estimate. BPB differences do not establish coherence.

In [ ]:
import statistics
rows = ["|Model|Historical validation BPB|Median measured32-token generation (s)|", "|---|---:|---:|"]
for m in generated["models"]:
    times = [r["seconds"] for r in generated["generations"] if r["model"] == m["id"]]
    rows.append(f"|{m['id'].upper()}|{m['historical_validation_bpb']:.8f}|{statistics.median(times):.3f}|")
display(Markdown("\n".join(rows)))

## 12. Real API, without public tunneling
An in-process FastAPI client calls canonical B01 generation and compares its decoded output with the shared generator. The deployed website is not required.

In [ ]:
api = section("api")
print(json.dumps(api["api"], indent=2, ensure_ascii=False))

## 13. Clean-process reload and final provenance
Every section launches a separate Python process and reloads verified packages. This final independent invocation confirms continued loading after earlier processes exit.

In [ ]:
run([PY, "-m", "tools.submission_demo", "--artifacts", WORK / "model_artifacts",
     "--section", "models", "--output", EVIDENCE / "restart"], cwd=WORK)
assert not run(["git", "status", "--porcelain"], cwd=WORK).strip()
summary = {"source_commit": environment["environment"]["source_commit"],
           "package_revision": download["revision"], "verified_models": [m["id"] for m in models["models"]],
           "generation_cases": len(generated["generations"]), "raw_ID_reference_cases": len(reference),
           "exact_raw_ID_matches": sum(r["reference_exact_ids"] for r in reference),
           "trained_attention_future_error": a["future_logit_max_abs_error"], "api_status": api["api"]["status"],
           "total_measured_notebook_seconds": time.perf_counter() - STARTED,
           "research_optimizer_updates": 0, "tiny_fixture_updates_per_test": 60}
(EVIDENCE / "summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

## 14. What this demonstrates — and its limits

We used the original implementation, verified the three exact selected packages,
inspected real trained attention, checked causality, generated authentic text and
called the genuine API. All receipts are in `outputs/submission-colab` for download
before the runtime expires. The model packages can be retrieved again by hash.

These models learned a small Shakespeare corpus with single-seed experiments.
F01/B01 prolonged training deteriorated on validation; selection uses the preserved
best states. Dialogue loops, malformed words and invalid-byte decoding remain real
limitations. Neither lower BPB nor a selected pleasing sample proves coherent
language. Outputs above are unedited. No test-set assessment, fine-tuning or new
research training occurred. The manual-attention/MLM-versus-CLM evidence audit,
final academic report and presentation are the **next separately reviewed phase**.